# Módulo 4: Primer modelo de clasificación

En el módulo anterior trabajamos con **regresión lineal**, donde el modelo intentaba predecir un número.

En este módulo damos el siguiente paso: aprenderemos a construir un modelo de **clasificación**, donde el modelo no predice una cantidad, sino una **categoría**.

Por ejemplo:

- ¿Aprueba o no aprueba?
- ¿Es spam o no es spam?
- ¿Pertenece a la clase A o a la clase B?
- ¿El resultado es positivo o negativo?

Este módulo está pensado para docentes que están comenzando en Machine Learning. La meta no es memorizar fórmulas, sino entender la idea general, entrenar un modelo sencillo y reflexionar sobre cómo podría usarse responsablemente en contextos educativos.


## Objetivos de aprendizaje

Al finalizar este módulo, podrás:

1. Explicar la diferencia entre regresión y clasificación.
2. Identificar cuándo un problema de Machine Learning es de clasificación.
3. Preparar variables de entrada `X` y variable objetivo `y`.
4. Entrenar un primer modelo de clasificación usando `LogisticRegression`.
5. Evaluar el modelo usando `accuracy` y matriz de confusión.
6. Reflexionar sobre los límites éticos de usar modelos predictivos en educación.


## 1. Regresión vs clasificación

| Tipo de modelo | ¿Qué predice? | Ejemplo |
|---|---|---|
| Regresión | Un número | Nota final, temperatura, ventas |
| Clasificación | Una categoría | Aprueba/No aprueba, Spam/No spam |

En el módulo anterior usamos regresión lineal para predecir valores numéricos.

Ahora usaremos clasificación para predecir una categoría.


## 2. Contexto educativo del ejemplo

Imaginemos que tenemos un conjunto de datos ficticio de estudiantes.

Para cada estudiante conocemos:

- horas de estudio por semana
- porcentaje de asistencia
- tareas entregadas
- participación en clase

También conocemos si el estudiante **aprobó** o **no aprobó**.

La pregunta será:

> ¿Puede un modelo aprender patrones básicos para clasificar si un estudiante probablemente aprueba o no aprueba?

Importante: este ejemplo es educativo. En la vida real, un modelo así no debería usarse para etiquetar estudiantes sin considerar el contexto humano, social, emocional y pedagógico.


In [ ]:
# Importamos las librerías necesarias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print("Librerías importadas correctamente.")

## 3. Creamos un dataset ficticio

Para esta primera actividad usaremos datos creados por nosotros. Esto permite controlar el ejemplo y enfocarnos en la idea central del modelo.

Luego, en una actividad futura, podríamos usar datos reales recolectados en el salón de clases, siempre respetando la privacidad de los estudiantes.


In [ ]:
# Para que los resultados sean reproducibles
np.random.seed(42)

# Número de estudiantes ficticios
n = 120

# Creamos variables simuladas
horas_estudio = np.random.randint(0, 16, n)
asistencia = np.random.randint(50, 101, n)
tareas_entregadas = np.random.randint(0, 11, n)
participacion = np.random.randint(1, 6, n)

# Creamos una regla sencilla para simular si aprueba o no.
# Esta regla NO es una verdad educativa; solo sirve para crear datos de práctica.
puntaje = (
    horas_estudio * 2.0 +
    asistencia * 0.25 +
    tareas_entregadas * 3.0 +
    participacion * 2.0 +
    np.random.normal(0, 6, n)
)

aprueba = np.where(puntaje >= 45, "Aprueba", "No aprueba")

df = pd.DataFrame({
    "horas_estudio": horas_estudio,
    "asistencia": asistencia,
    "tareas_entregadas": tareas_entregadas,
    "participacion": participacion,
    "resultado": aprueba
})

df.head()

## 4. Exploramos los datos

Antes de entrenar un modelo, siempre debemos mirar los datos.

Preguntas importantes:

- ¿Cuántos ejemplos tenemos?
- ¿Qué columnas aparecen?
- ¿Hay más estudiantes en una categoría que en otra?
- ¿Los datos parecen razonables?


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["resultado"].value_counts()

## 5. Visualizamos los datos

Una visualización nos ayuda a entender patrones antes de usar el modelo.

Veamos la relación entre horas de estudio, asistencia y resultado.


In [ ]:
plt.figure(figsize=(8, 5))

for resultado in df["resultado"].unique():
    subset = df[df["resultado"] == resultado]
    plt.scatter(
        subset["horas_estudio"],
        subset["asistencia"],
        label=resultado,
        alpha=0.7
    )

plt.xlabel("Horas de estudio por semana")
plt.ylabel("Asistencia (%)")
plt.title("Relación entre estudio, asistencia y resultado")
plt.legend()
plt.show()

## 6. Preparamos `X` y `y`

En Machine Learning supervisado usamos:

- `X`: variables de entrada, también llamadas características o *features*.
- `y`: respuesta que queremos predecir.

En este ejemplo:

`X` será:

- horas de estudio
- asistencia
- tareas entregadas
- participación

`y` será:

- resultado: Aprueba o No aprueba


In [ ]:
X = df[["horas_estudio", "asistencia", "tareas_entregadas", "participacion"]]
y = df["resultado"]

print("Variables de entrada X:")
display(X.head())

print("Variable objetivo y:")
display(y.head())

## 7. Dividimos los datos en entrenamiento y prueba

No queremos evaluar el modelo con los mismos datos que usó para aprender.

Por eso dividimos el dataset en dos partes:

- **Entrenamiento:** datos que el modelo usa para aprender.
- **Prueba:** datos nuevos para evaluar si el modelo aprendió bien.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)

## 8. Entrenamos nuestro primer modelo de clasificación

Usaremos un modelo llamado **Regresión Logística**.

Aunque su nombre incluye la palabra "regresión", se usa comúnmente para problemas de clasificación.

En este caso, el modelo intentará aprender patrones para clasificar estudiantes en:

- Aprueba
- No aprueba


In [ ]:
modelo = LogisticRegression(max_iter=1000)

modelo.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

## 9. Hacemos predicciones

Ahora usamos el modelo para predecir el resultado de estudiantes que no vio durante el entrenamiento.


In [ ]:
y_pred = modelo.predict(X_test)

resultados = pd.DataFrame({
    "Real": y_test.values,
    "Predicción": y_pred
})

resultados.head(10)

## 10. Evaluamos el modelo con accuracy

El **accuracy** indica qué proporción de predicciones fueron correctas.

Por ejemplo, un accuracy de 0.80 significa que el modelo acertó el 80% de las veces.

Sin embargo, accuracy no siempre cuenta toda la historia. Por eso también veremos la matriz de confusión.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy del modelo: {accuracy:.2f}")

## 11. Matriz de confusión

La matriz de confusión nos permite ver:

- cuántas veces el modelo acertó
- cuántas veces se equivocó
- qué tipo de errores cometió

Esto es muy importante en educación, porque no todos los errores tienen el mismo impacto.


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=modelo.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=modelo.classes_
)

disp.plot()
plt.title("Matriz de confusión")
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))

## 12. Probemos con un nuevo estudiante

Ahora podemos crear un ejemplo nuevo y pedirle al modelo que clasifique.

Supongamos un estudiante con:

- 6 horas de estudio por semana
- 82% de asistencia
- 7 tareas entregadas
- participación 3 de 5


In [ ]:
nuevo_estudiante = pd.DataFrame({
    "horas_estudio": [6],
    "asistencia": [82],
    "tareas_entregadas": [7],
    "participacion": [3]
})

prediccion = modelo.predict(nuevo_estudiante)
probabilidades = modelo.predict_proba(nuevo_estudiante)

print("Predicción:", prediccion[0])

pd.DataFrame(
    probabilidades,
    columns=modelo.classes_
)

## 13. Actividad para docentes

Modifica los valores del estudiante anterior y observa cómo cambia la predicción.

Prueba ejemplos como:

1. Estudiante con alta asistencia, pero pocas tareas entregadas.
2. Estudiante con muchas horas de estudio, pero baja participación.
3. Estudiante con baja asistencia, pero muchas tareas entregadas.
4. Estudiante con valores altos en todas las variables.
5. Estudiante con valores bajos en todas las variables.

### Preguntas de reflexión

- ¿Qué variable parece influir más?
- ¿El resultado del modelo siempre parece justo?
- ¿Qué información importante no está en el dataset?
- ¿Qué riesgo habría si una escuela usara un modelo así sin criterio docente?


## 14. Discusión ética: cuidado con etiquetar estudiantes

Este ejemplo nos ayuda a entender clasificación, pero también nos recuerda algo muy importante:

> Un estudiante no puede reducirse a cuatro columnas de datos.

Variables como horas de estudio, asistencia o tareas entregadas pueden ser útiles, pero no explican toda la realidad.

Un estudiante puede tener:

- problemas familiares
- dificultades económicas
- responsabilidades en casa
- barreras de idioma
- ansiedad
- falta de acceso a internet
- necesidades educativas especiales
- situaciones de salud

Por eso, en educación, los modelos de Machine Learning deben usarse como herramientas de apoyo, no como jueces finales.


## 15. Mini-proyecto del módulo

Diseña una idea de clasificación que podrías usar en tu materia.

Tu propuesta debe incluir:

- Materia
- Grado
- Pregunta de clasificación
- Variables de entrada
- Categoría que quieres predecir
- Posibles beneficios
- Posibles riesgos
- Cómo asegurarías un uso ético

### Ejemplos

| Materia | Pregunta de clasificación |
|---|---|
| Ciencias | ¿Una muestra de agua es segura o no segura? |
| Matemáticas | ¿Un estudiante necesita repaso adicional o no? |
| Biología | ¿Una planta pertenece a una especie u otra? |
| Tecnología | ¿Un mensaje parece spam o no spam? |
| Física | ¿Un movimiento es uniforme o acelerado? |


## 16. Cierre

En este módulo aprendimos que:

- La clasificación predice categorías.
- La regresión predice números.
- Un modelo puede aprender patrones a partir de ejemplos.
- `LogisticRegression` puede usarse como primer modelo de clasificación.
- El accuracy ayuda a evaluar, pero no es suficiente.
- La matriz de confusión permite analizar los errores.
- En educación, los modelos deben usarse con responsabilidad y criterio humano.

### Idea final

Machine Learning puede ayudar a los docentes a explorar datos, detectar patrones y generar preguntas.  
Pero la decisión pedagógica siempre debe seguir estando guiada por el maestro.


## Extensión opcional: Cambiar el contexto

Puedes cambiar el dataset ficticio por otro contexto:

- clasificación de flores
- clasificación de tipos de movimiento
- clasificación de mensajes
- clasificación de materiales
- clasificación de datos ambientales de Puerto Rico

Lo importante es mantener la pregunta central:

> ¿Queremos predecir una categoría?

Si la respuesta es sí, estamos ante un problema de clasificación.
